In [ ]:
# ===================== Imports =====================
import argparse, sys, os, glob, json, math, datetime, torch, yaml
import matplotlib.pyplot as plt, numpy as np, pandas as pd

from collections import defaultdict
from scipy.stats import norm, pearsonr

from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LinearRegression
from sklearn.utils import resample
from pathlib import Path

sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/src/model/')
sys.path.append("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/")

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

from encoders import *
from utls.plotting import ensure_dir
from utls.loading import load_results_with_exclusion_2, move_sequences_to_used, load_results_with_exclusion_no_dropping
from utls.runners import run_experiment_scores
from utls.runners_v2 import (
    run_experiment_grid,
    run_experiment_scores,
    run_experiment_scores_itemwise,
    run_experiment_itemwise_hits_fas,
    make_noise_schedule
)

from utls.analysis_helpers import rocs_across_noise, convert_human_to_model_struct, compute_scaling_vs_human, convert_human_to_model_struct
from utls.analysis_helpers import auroc_to_dprime, compute_model_dprime_curve
from utls.analysis_helpers import roc_for_isi, auroc_to_dprime
from utls.plotting import plot_across_noise, plot_noise_overlays
from utls.io_utils import make_model_save_dir, save_all_figures, save_single_figure, save_runs_summary
from utls.roc_utils import roc_from_arrays 
from utls.runners_utils import *


def load_config(cfg_path=None):
    """
    Load YAML config.
    Priority:
      1. cfg_path argument (if provided)
      2. sys.argv[1]
    """
    if cfg_path is None:
        if len(sys.argv) != 2:
            raise RuntimeError("Usage: python main.py path/to/run.yaml")
        cfg_path = sys.argv[1]

    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(cfg_path)

    with open(cfg_path, "r") as f:
        return yaml.safe_load(f), cfg_path


tasks = {
    0: "ind-nature",
    1: "global-music",
    2: "atexts",
}

In [ ]:
model_cfg, model_cfg_path = load_config("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/model_yamls/test/run_000012.yaml")

noise_cfg = model_cfg["noise_model"]
model_cfg

assert "t_step" in noise_cfg, "t_step is needed for two-regime"

In [ ]:
# ---------------------------
# SETTINGS (from YAML)
# ---------------------------
# ---- experiment ----
exp_cfg = model_cfg["experiment"]

which_task = exp_cfg["which_task"]
is_multi   = exp_cfg["is_multi"]
which_isi  = exp_cfg.get("which_isi", None)

if is_multi:
    isis = [0, 1, 2, 4, 8, 16, 32, 64]
else:
    assert which_isi is not None, "which_isi required if not multi-ISI"
    isis = [0, which_isi]

# ---- metric ----
metric = model_cfg["metric"]

# ---- noise model ----
noise_cfg = model_cfg["noise_model"]
noise_mode = noise_cfg["name"]
print(noise_cfg)

# ---- representation ----
repr_cfg = model_cfg["representation"]

encoder_type = repr_cfg["type"]
layer   = repr_cfg.get("layer", None)
PC_DIMS = repr_cfg.get("pc_dims", None)

# ---------------------------
# 1. Load data
# ---------------------------
exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = load_experiment_data(
    which_task, which_isi, is_multi)

# ---------------------------
# 2. Human curve
# ---------------------------
human_curve = compute_human_curve(human_runs, is_multi, which_isi)
results_root = model_cfg["results_root"]
tag = model_cfg.get("tag", "untagged")

if noise_mode == "two-regime":
    assert "t_step" in noise_cfg, "two-regime requires t_step"
    t_step = noise_cfg["t_step"]
    noise_tag = f"{noise_mode}_t{t_step}"
else:
    noise_tag = noise_mode

save_figs = (
    f"{results_root}/"
    f"figures/test/"
    f"{task_name}/"
    f"{encoder_type}/"
    f"{metric}/"
    f"{noise_tag}/"
)

save_fits = f"{results_root}/test/fits"


print(save_figs, save_fits)
ensure_dir(save_figs)

ensure_dir(save_fits)


In [ ]:
encoder_type = repr_cfg["type"]     # e.g. "resnet50"
layer        = repr_cfg.get("layer")
pc_dims      = repr_cfg.get("pc_dims")

NN_ENCODERS = {"kell2018", "resnet50"}

encoder_task = (
    "word_speaker_audioset"
    if encoder_type in NN_ENCODERS
    else "audioset"
)

encoder_cfg = dict(
    encoder_type=encoder_type,      # e.g. "resnet50"
    model_name=encoder_type,        # same by design
    task=encoder_task,
    statistics_dict=statistics_set.statistics,
    model_params=model_params,
    pc_dims=pc_dims,
    sr=20000,
    duration=2.0,
    rms_level=0.05,
    device="cuda",
)

# ---- representation-specific fields ----
if encoder_type in ("kell2018", "resnet50"):
    encoder_cfg["layer"] = layer
    assert layer is not None, f"layer required for {encoder_type}"

if encoder_type == "texture":
    encoder_cfg["pc_dims"] = pc_dims
    assert pc_dims is not None, "pc_dims required for texture encoder"

encoder_name = make_encoder_name(encoder_cfg)
print("Encoder name:", encoder_name)

encoder = build_encoder(encoder_cfg)
X = encode_stimuli(encoder, all_files)


noise_cfg = model_cfg["noise_model"]
noise_mode = noise_cfg["name"]

# ---------------------------
# Noise parameter bounds (from YAML)
# ---------------------------

param_bounds = {
    "sigma0": (noise_cfg["sigma0_min"], noise_cfg["sigma0_max"]),
}

if noise_mode == "two-regime":
    param_bounds["sigma1"] = (
        noise_cfg["sigma1_min"],
        noise_cfg["sigma1_max"],
    )
    param_bounds["t_step"] = (
        noise_cfg["t_step"],
        noise_cfg["t_step"],
    )


opt_results = random_search_gridlike(
    n_samples=model_cfg["experiment"]["n_samples"],
    param_bounds=param_bounds,
    noise_mode=noise_mode,
    metric=metric,
    X0=X,
    name_to_idx=name_to_idx,
    experiment_list=exp_list[:model_cfg["experiment"]["n_seqs"]],
    isis=isis,
    human_curve=human_curve,
    layer=encoder_name,          # legacy arg, keep for now
    encoder_name=encoder_name,   # canonical identifier
    hr_task_name=hr_task_name,
    debug=False,
)


In [ ]:
# import random
# opt_results_all = []

# dims = [100, 256]
# n = 40
# n_samples = 10
# age=3

# for pc_dim in dims:
#     cfg = dict(
#         encoder_type="texture_pca",
#         model_name="texture_pca",
#         #layer=layer,
#         task="v1",
#         statistics_dict=statistics_set.statistics,
#         pc_dims=pc_dim,
#         model_params=model_params,
#         sr=20000,
#         duration=2.0,
#         rms_level=0.05,
#         device="cuda",
#     )
    
#     encoder_name = make_encoder_name(cfg)
#     print("Encoder name:", encoder_name)
    
#     encoder = build_encoder(cfg)
#     X = encode_stimuli(encoder, all_files)
    
#     opt_results = random_search_gridlike(
#         n_samples=n_samples,
#         param_bounds = {
#             "sigma0": (0.25, 1),
#             # "sigma1": (0.0, .25),
#             # "t_step": (age, age),
#         },
#         noise_mode=noise_mode,
#         metric="cosine",
#         X0=X,
#         name_to_idx=name_to_idx,
#         experiment_list=random.sample(exp_list, n),
#         isis=isis,
#         human_curve=human_curve,
#         layer=encoder_name,
#         encoder_name=encoder_name,
#         hr_task_name=hr_task_name,
#         debug=False,
#     )
    
#     opt_results_all += opt_results

In [ ]:

plot_best_model_histograms(best_fits, isis, save_figs)